In [1]:
import time
import requests
import pandas as pd
from pathlib import Path
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

main_dir = Path("C:/Users/gabri/Documents/Data_projects/MCcharactertization/ROSMAP/Top_sets/Set309")
#main_dir = Path('/Users/ga0028vi/Documents/MCcharactertization/ROSMAP/Top_sets/Set309')

infile = Path(main_dir, "_set_data/PLSsig_genelists_long.csv")

out_dir =Path(main_dir, "STRING_AGM_analysis/Annotations")
out_dir.mkdir(parents=True, exist_ok=True)

agm_df = pd.read_csv(infile)
print(agm_df.head())

                                  AGM_ID        gene
0  Excitatory_ExcL45RORBGABRG1_Cluster_0       ISG15
1  Excitatory_ExcL45RORBGABRG1_Cluster_0     FAM131C
2  Excitatory_ExcL45RORBGABRG1_Cluster_0  AL445685.1
3  Excitatory_ExcL45RORBGABRG1_Cluster_0    ARHGAP29
4  Excitatory_ExcL45RORBGABRG1_Cluster_0     C1orf61


## Ensembl queries

In [2]:
# query Ensembl in batches
def query_ensembl_gene_biotypes(
    genes,
    species="homo_sapiens",
    batch_size=500,
    sleep_time=0.2,
):
    """
    Query Ensembl REST API for gene biotypes using gene symbols.

    Parameters
    ----------
    genes : list-like
        Gene symbols, e.g. ["ISG15", "AL445685.1"].
    species : str
        Ensembl species name. For human, use "homo_sapiens".
    batch_size : int
        Number of symbols per API request.
    sleep_time : float
        Pause between requests to be polite to the API.

    Returns
    -------
    pd.DataFrame
        Annotation table with one row per returned gene.
    """

    genes = (
        pd.Series(genes)
        .dropna()
        .astype(str)
        .str.strip()
        .unique()
        .tolist()
    )

    url = f"https://rest.ensembl.org/lookup/symbol/{species}"

    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json"
    }

    all_rows = []

    for i in range(0, len(genes), batch_size):
        batch = genes[i:i + batch_size]

        payload = {
            "symbols": batch
        }

        response = requests.post(
            url,
            headers=headers,
            json=payload,
            timeout=30
        )

        if response.status_code != 200:
            print(f"Batch {i // batch_size + 1} failed with status {response.status_code}")
            print(response.text[:500])
            continue

        data = response.json()

        for query_symbol, hit in data.items():
            if hit is None:
                continue

            all_rows.append({
                "gene": query_symbol,
                "ensembl_gene_id": hit.get("id"),
                "display_name": hit.get("display_name"),
                "description": hit.get("description"),
                "biotype": hit.get("biotype"),
                "object_type": hit.get("object_type"),
                "source": hit.get("source"),
                "seq_region_name": hit.get("seq_region_name"),
                "start": hit.get("start"),
                "end": hit.get("end"),
                "strand": hit.get("strand")
            })

        print(f"Queried {min(i + batch_size, len(genes))}/{len(genes)} genes")
        time.sleep(sleep_time)

    annot_df = pd.DataFrame(all_rows)

    return annot_df

In [3]:
# Query unique genes
unique_genes = agm_df["gene"].dropna().unique()

annot_df = query_ensembl_gene_biotypes(unique_genes)

print(annot_df["biotype"].value_counts(dropna=False))


# Merge annotation back to AGM table
agm_annotated_df = agm_df.merge(annot_df, on="gene", how="left")
agm_annotated_df["annotation_found"] = agm_annotated_df["ensembl_gene_id"].notna()

# Save the intermediary file from ensembl with the OG gene names
outfile = Path(out_dir, "AGM_gene_list_long_with_biotypes.csv")
agm_annotated_df.to_csv(outfile, index=False)
print(f"Saved: {outfile}")

# Save missing gene annotations for HGNC
miss_file = Path(out_dir, "AGM_genes_missing_annotation.csv")
missing_df = (agm_annotated_df.loc[~agm_annotated_df["annotation_found"], ["gene"]].drop_duplicates().sort_values("gene"))
missing_df.to_csv(miss_file, index=False)
print(f"Saved: {miss_file}")



Queried 500/2004 genes
Queried 1000/2004 genes
Queried 1500/2004 genes
Queried 2000/2004 genes
Queried 2004/2004 genes
biotype
protein_coding    1613
lncRNA             128
TR_C_gene            1
Name: count, dtype: int64
Saved: C:\Users\gabri\Documents\Data_projects\MCcharactertization\ROSMAP\Top_sets\Set309\STRING_AGM_analysis\Annotations\AGM_gene_list_long_with_biotypes.csv
Saved: C:\Users\gabri\Documents\Data_projects\MCcharactertization\ROSMAP\Top_sets\Set309\STRING_AGM_analysis\Annotations\AGM_genes_missing_annotation.csv


# Cleaning up the gene list

In [19]:
# First-pass Ensembl annotation
ensembl_file = out_dir / "AGM_gene_list_long_with_biotypes.csv"
# HGNC symbol checker output
hgnc_file = out_dir / "hgnc-symbol-check.csv"

hgnc_df = pd.read_csv(hgnc_file, skiprows=1)
agm_df = pd.read_csv(infile)
ensembl_df = pd.read_csv(ensembl_file)


In [20]:
ensembl_df = ensembl_df.rename(columns={"gene": "gene_og"})
ensembl_df["gene_renamed"] = ensembl_df["gene_og"]


#keep only HGNC rows where Input is not 0
hgnc_df = hgnc_df[hgnc_df["Input"] != 0].copy()


# merge HGNC Approved symbol onto ensembl_df
ensembl_df = ensembl_df.merge(hgnc_df[["Input", "Approved symbol"]],
                              left_on="gene_renamed",
                              right_on="Input",
                              how="left")

#Replace gene_renamed with HGNC Approved symbol
ensembl_df.loc[ensembl_df["Approved symbol"].notna(),"gene_renamed"] = ensembl_df.loc[ensembl_df["Approved symbol"].notna(),"Approved symbol"]


# bool if HGNC changed the gene name
ensembl_df["gene_name_changed_by_hgnc"] = (ensembl_df["gene_og"] != ensembl_df["gene_renamed"])

# Drop temp columns
ensembl_df = ensembl_df.drop(columns=["Input", "Approved symbol"])

In [21]:
genes_to_rerun = (ensembl_df.loc[(~ensembl_df["annotation_found"]) 
                                 & (ensembl_df["gene_name_changed_by_hgnc"]),"gene_renamed"]
                  .drop_duplicates().tolist())

In [23]:
# Rerun ENSEMBL
# genes that failed first-pass Ensembl annotation,
# but now have an HGNC-updated symbol
print("Genes to re-query:", len(genes_to_rerun))
print(genes_to_rerun[:10])

rerun_annot_df = query_ensembl_gene_biotypes(genes_to_rerun)

print(rerun_annot_df.head())
print(rerun_annot_df["biotype"].value_counts(dropna=False))

Genes to re-query: 43
['MIR9-1HG', 'NICOL1', 'MARCHF1', 'TAFA2', 'CCN2', 'CCN2-AS1', 'ABTB3', 'MYL11', 'NHERF1', 'H2AC6']
Queried 43/43 genes
      gene  ensembl_gene_id display_name  \
0   NHERF1  ENSG00000109062       NHERF1   
1     BBLN  ENSG00000171159         BBLN   
2    CCDST  ENSG00000236427        CCDST   
3  CAV2-DT  ENSG00000237813      CAV2-DT   
4    AFG2A  ENSG00000145375        AFG2A   

                                         description         biotype  \
0  NHERF family PDZ scaffold protein 1 [Source:HG...  protein_coding   
1  bublin coiled coil protein [Source:HGNC Symbol...  protein_coding   
2  cervical cancer associated DHX9 suppressive tr...          lncRNA   
3  CAV2 divergent transcript [Source:HGNC Symbol;...          lncRNA   
4  AFG2 AAA ATPase homolog A [Source:HGNC Symbol;...  protein_coding   

  object_type          source seq_region_name      start        end  strand  
0        Gene  ensembl_havana              17   74748619   74769360       1  
1   

In [24]:
# make copy just in case
updated_df = ensembl_df.copy()
rerun_tmp = rerun_annot_df.rename(columns={"gene": "gene_renamed"}).copy()

In [25]:
annot_cols = ["ensembl_gene_id", "display_name", "description", "biotype", "object_type",
              "source", "seq_region_name", "start", "end", "strand"]

In [26]:
#set index to gene_renamed so matching happens by the renamed HGNC symbol
updated_df = updated_df.set_index("gene_renamed")
rerun_tmp = rerun_tmp.set_index("gene_renamed")

#replace annotation columns if rerun annotation exists
updated_df.update(rerun_tmp[annot_cols])

#bring gene_renamed back as a normal column
updated_df = updated_df.reset_index()

In [27]:
#update the annotation if
updated_df["annotation_found"] = updated_df["ensembl_gene_id"].notna()

In [28]:
# Add ncRNA / protein-coding flags
lncRNA_biotypes = {
    "lncRNA",
    "lincRNA",
    "antisense",
    "sense_intronic",
    "sense_overlapping",
    "processed_transcript",
    "3prime_overlapping_ncRNA",
    "macro_lncRNA",
    "bidirectional_promoter_lncRNA",
    "non_coding"
}

small_ncRNA_biotypes = {
    "miRNA",
    "snRNA",
    "snoRNA",
    "rRNA",
    "misc_RNA",
    "scaRNA",
    "scRNA",
    "ribozyme",
    "sRNA",
    "vault_RNA",
    "Mt_rRNA",
    "Mt_tRNA"
}

ncRNA_biotypes = lncRNA_biotypes.union(small_ncRNA_biotypes)

updated_df["is_lncRNA"] = updated_df["biotype"].isin(lncRNA_biotypes)
updated_df["is_any_ncRNA"] = updated_df["biotype"].isin(ncRNA_biotypes)
updated_df["is_protein_coding"] = updated_df["biotype"].eq("protein_coding")

In [29]:
cols_order = ["AGM_ID",
              "gene_og",
              "gene_renamed",
              "gene_name_changed_by_hgnc",
              "ensembl_gene_id",
              "display_name",
              "description",
              "biotype",
              "is_lncRNA",
              "is_any_ncRNA",
              "is_protein_coding",
              "object_type",
              "source",
              "seq_region_name",
              "start",
              "end",
              "strand",
              "annotation_found"
              ]

updated_df = updated_df[cols_order].copy()

In [30]:
print("\nAnnotation found:")
print(updated_df["annotation_found"].value_counts(dropna=False))

print("\nBiotypes:")
print(updated_df["biotype"].value_counts(dropna=False).head(30))

print("\nHGNC renamed genes:")
print(updated_df["gene_name_changed_by_hgnc"].value_counts(dropna=False))

still_missing = (
    updated_df
    .loc[~updated_df["annotation_found"], "gene_og"]
    .dropna()
    .drop_duplicates()
    .sort_values()
)

print("\nStill missing:", len(still_missing))
print(still_missing.head(100).to_list())


Annotation found:
annotation_found
True     2364
False     312
Name: count, dtype: int64

Biotypes:
biotype
protein_coding    2181
NaN                312
lncRNA             182
TR_C_gene            1
Name: count, dtype: int64

HGNC renamed genes:
gene_name_changed_by_hgnc
False    2616
True       60
Name: count, dtype: int64

Still missing: 221
['AC003988.1', 'AC003991.1', 'AC004158.1', 'AC004594.1', 'AC004852.2', 'AC004949.1', 'AC005064.1', 'AC005154.5', 'AC005537.1', 'AC005726.5', 'AC005828.3', 'AC005842.1', 'AC006059.1', 'AC006148.1', 'AC006305.1', 'AC006450.3', 'AC007091.1', 'AC007319.1', 'AC007344.1', 'AC007402.1', 'AC007614.1', 'AC007656.1', 'AC007785.1', 'AC008014.1', 'AC008105.3', 'AC008415.1', 'AC008574.1', 'AC008691.1', 'AC008780.1', 'AC009063.2', 'AC009163.1', 'AC009226.1', 'AC009501.1', 'AC010099.4', 'AC010127.1', 'AC010260.1', 'AC010967.1', 'AC011287.1', 'AC011468.3', 'AC012593.1', 'AC016152.1', 'AC016597.1', 'AC016687.2', 'AC016716.2', 'AC016722.2', 'AC018682.1', 'AC0190

In [31]:
# Final output
final_file = out_dir / "AGM_gene_list_long_with_biotypes_HGNC_rescued.csv"
updated_df.to_csv(final_file, index=False)

In [33]:
updated_df['is_unannotated'] = ~updated_df["annotation_found"]

In [35]:
gsea_gene_lists = (
    updated_df
    .loc[updated_df["annotation_found"], ["AGM_ID", "gene_renamed"]]
    .dropna()
    .drop_duplicates()
    .sort_values(["AGM_ID", "gene_renamed"])
)

gsea_gene_lists.to_csv(Path(out_dir, "PLSgenelist_long_HGNCcorrected.csv"), index=False)

## Analysis of enrichment

In [38]:
category = "is_lncRNA"

all_genes = set(updated_df["gene_og"].dropna().unique())

In [ ]:
len(all_genes)

gene_status = (
    updated_df[["gene_og", category]]
    .drop_duplicates("gene_og")
    .set_index("gene_og")[category]
    .to_dict()
)


{'ISG15': False,
 'FAM131C': False,
 'AL445685.1': False,
 'ARHGAP29': False,
 'C1orf61': True,
 'PCP4L1': False,
 'CHI3L1': False,
 'GCSAML': False,
 'HPCAL1': False,
 'LINC00276': True,
 'RASGRP3': False,
 'EPAS1': False,
 'LINC01798': True,
 'ARHGAP15': False,
 'GAD1': False,
 'LINC01090': True,
 'PARD3B': False,
 'TMBIM1': False,
 'SLC6A1-AS1': True,
 'FEZF2': False,
 'AC026316.5': False,
 'IGF2BP2': False,
 'SST': False,
 'FAM114A1': False,
 'RBM47': False,
 'REST': False,
 'NDST4': False,
 'LINC01099': True,
 'LINC01194': True,
 'MYO10': False,
 'FYB1': False,
 'AC114947.2': False,
 'HSPB3': False,
 'SETD9': False,
 'S100Z': False,
 'HOMER1': False,
 'ISOC1': False,
 'FGF1': False,
 'GPX3': False,
 'FAT2': False,
 'LCP2': False,
 'KCNIP1': False,
 'TUBB2B': False,
 'GCNT2': False,
 'SOX4': False,
 'HLA-A': False,
 'HLA-B': False,
 'HLA-DPA1': False,
 'TENT5A': False,
 'PKIB': False,
 'FABP7': False,
 'SAMD3': False,
 'UTRN': False,
 'AL160162.1': False,
 'AC005154.5': False,
 'EL

In [ ]:
category = "is_lncRNA"

all_genes = set(updated_df["gene_og"].dropna().unique())

gene_status = (
    updated_df[["gene_og", category]]
    .drop_duplicates("gene_og")
    .set_index("gene_og")[category]
    .to_dict()
)

results = []

for agm_id in updated_df["AGM_ID"].dropna().unique():

    agm_genes = set(
        updated_df.loc[updated_df["AGM_ID"] == agm_id, "gene_og"]
        .dropna()
        .unique()
    )

    other_genes = all_genes - agm_genes

    category_in_agm = sum(gene_status[g] for g in agm_genes)
    not_category_in_agm = len(agm_genes) - category_in_agm

    category_outside = sum(gene_status[g] for g in other_genes)
    not_category_outside = len(other_genes) - category_outside

    table = [
        [category_in_agm, not_category_in_agm],
        [category_outside, not_category_outside]
    ]

    odds_ratio, pval = fisher_exact(table, alternative="greater")

    results.append({
        "AGM_ID": agm_id,
        "category": category,
        "n_genes": len(agm_genes),
        "n_category": category_in_agm,
        "prop_category": category_in_agm / len(agm_genes),
        "background_prop_category": category_outside / len(other_genes),
        "odds_ratio": odds_ratio,
        "pval": pval
    })

res_df = pd.DataFrame(results)
res_df["FDR"] = multipletests(res_df["pval"], method="fdr_bh")[1]
res_df = res_df.sort_values("FDR")
res_df

,AGM_ID,category,n_genes,n_category,prop_category,background_prop_category,odds_ratio,pval,FDR
15,Inhibitory_InhLAMP5NRG1Rosehip_Cluster_6,is_lncRNA,260,38,0.146154,0.060206,2.671900,0.000004,0.000099
16,Inhibitory_InhLAMP5NRG1Rosehip_Cluster_7,is_lncRNA,93,14,0.150538,0.067504,2.448042,0.005052,0.060626
14,Inhibitory_InhLAMP5NRG1Rosehip_Cluster_12,is_lncRNA,46,7,0.152174,0.069459,2.404600,0.041396,0.331170
19,Oligodendrocytes_Oli_Cluster_12,is_lncRNA,92,10,0.108696,0.069561,1.631212,0.115105,0.690632
1,Excitatory_ExcL45RORBGABRG1_Cluster_7,is_lncRNA,78,8,0.102564,0.070093,1.516190,0.188046,0.902620
0,Excitatory_ExcL45RORBGABRG1_Cluster_0,is_lncRNA,144,12,0.083333,0.070430,1.199861,0.328024,1.000000
21,Immune_Micall_Cluster_11,is_lncRNA,203,6,0.029557,0.076069,0.369928,0.997869,1.000000
20,Oligodendrocytes_Oli_Cluster_3,is_lncRNA,136,9,0.066176,0.071734,0.917029,0.647610,1.000000
18,Astrocytes_AstDPP10_Cluster_6,is_lncRNA,98,8,0.081633,0.070829,1.166091,0.399803,1.000000
17,OPCs_OPC_Cluster_16,is_lncRNA,76,3,0.039474,0.072614,0.524853,0.918816,1.000000


In [ ]:

category = "is_unannotated"

all_genes = set(updated_df["gene_og"].dropna().unique())

gene_status = (
    updated_df[["gene_og", category]]
    .drop_duplicates("gene_og")
    .set_index("gene_og")[category]
    .to_dict()
)

results = []

for agm_id in updated_df["AGM_ID"].dropna().unique():

    agm_genes = set(
        updated_df.loc[updated_df["AGM_ID"] == agm_id, "gene_og"]
        .dropna()
        .unique()
    )

    other_genes = all_genes - agm_genes

    category_in_agm = sum(gene_status[g] for g in agm_genes)
    not_category_in_agm = len(agm_genes) - category_in_agm

    category_outside = sum(gene_status[g] for g in other_genes)
    not_category_outside = len(other_genes) - category_outside

    table = [
        [category_in_agm, not_category_in_agm],
        [category_outside, not_category_outside]
    ]

    odds_ratio, pval = fisher_exact(table, alternative="greater")

    results.append({
        "AGM_ID": agm_id,
        "category": category,
        "n_genes": len(agm_genes),
        "n_category": category_in_agm,
        "prop_category": category_in_agm / len(agm_genes),
        "background_prop_category": category_outside / len(other_genes),
        "odds_ratio": odds_ratio,
        "pval": pval
    })

res_df = pd.DataFrame(results)
res_df["FDR"] = multipletests(res_df["pval"], method="fdr_bh")[1]
res_df = res_df.sort_values("FDR")
res_df

,AGM_ID,category,n_genes,n_category,prop_category,background_prop_category,odds_ratio,pval,FDR
1,Excitatory_ExcL45RORBGABRG1_Cluster_7,is_unannotated,78,22,0.282051,0.103323,3.409368,0.000016,0.000375
15,Inhibitory_InhLAMP5NRG1Rosehip_Cluster_6,is_unannotated,260,48,0.184615,0.099197,2.056058,0.000085,0.001018
19,Oligodendrocytes_Oli_Cluster_12,is_unannotated,92,22,0.239130,0.104079,2.705384,0.000244,0.001727
16,Inhibitory_InhLAMP5NRG1Rosehip_Cluster_7,is_unannotated,93,22,0.236559,0.104134,2.665723,0.000288,0.001727
11,Inhibitory_InhG7_Cluster_10,is_unannotated,57,12,0.210526,0.107345,2.217544,0.018493,0.088765
14,Inhibitory_InhLAMP5NRG1Rosehip_Cluster_12,is_unannotated,46,9,0.195652,0.108274,2.003315,0.059183,0.236732
7,Excitatory_ExcL35RORBPLCH1_Cluster_14,is_unannotated,221,30,0.135747,0.107123,1.309175,0.122864,0.421248
17,OPCs_OPC_Cluster_16,is_unannotated,76,11,0.144737,0.108921,1.384469,0.209342,0.628027
22,Immune_Micall_Cluster_17,is_unannotated,85,11,0.129412,0.109432,1.209717,0.332273,0.776072
0,Excitatory_ExcL45RORBGABRG1_Cluster_0,is_unannotated,144,17,0.118056,0.109677,1.086614,0.420372,0.776072


In [22]:
category = "gene_name_changed_by_hgnc"

all_genes = set(updated_df["gene_og"].dropna().unique())

gene_status = (
    updated_df[["gene_og", category]]
    .drop_duplicates("gene_og")
    .set_index("gene_og")[category]
    .to_dict()
)

results = []

for agm_id in updated_df["AGM_ID"].dropna().unique():

    agm_genes = set(
        updated_df.loc[updated_df["AGM_ID"] == agm_id, "gene_og"]
        .dropna()
        .unique()
    )

    other_genes = all_genes - agm_genes

    category_in_agm = sum(gene_status[g] for g in agm_genes)
    not_category_in_agm = len(agm_genes) - category_in_agm

    category_outside = sum(gene_status[g] for g in other_genes)
    not_category_outside = len(other_genes) - category_outside

    table = [
        [category_in_agm, not_category_in_agm],
        [category_outside, not_category_outside]
    ]

    odds_ratio, pval = fisher_exact(table, alternative="greater")

    results.append({
        "AGM_ID": agm_id,
        "category": category,
        "n_genes": len(agm_genes),
        "n_category": category_in_agm,
        "prop_category": category_in_agm / len(agm_genes),
        "background_prop_category": category_outside / len(other_genes),
        "odds_ratio": odds_ratio,
        "pval": pval
    })

res_df = pd.DataFrame(results)
res_df["FDR"] = multipletests(res_df["pval"], method="fdr_bh")[1]
res_df = res_df.sort_values("FDR")
res_df

,AGM_ID,category,n_genes,n_category,prop_category,background_prop_category,odds_ratio,pval,FDR
19,Oligodendrocytes_Oli_Cluster_12,gene_name_changed_by_hgnc,92,5,0.054348,0.019351,2.912395,0.040570,0.973686
20,Oligodendrocytes_Oli_Cluster_3,gene_name_changed_by_hgnc,136,4,0.029412,0.020343,1.459330,0.317438,0.974946
2,Excitatory_ExcL45RORBGABRG1_Cluster_8,gene_name_changed_by_hgnc,63,2,0.031746,0.020608,1.558197,0.383261,0.974946
4,Excitatory_ExcL45RORBGABRG1_Cluster_5,gene_name_changed_by_hgnc,199,5,0.025126,0.020499,1.231541,0.406227,0.974946
18,Astrocytes_AstDPP10_Cluster_6,gene_name_changed_by_hgnc,98,3,0.030612,0.020462,1.511741,0.338228,0.974946
15,Inhibitory_InhLAMP5NRG1Rosehip_Cluster_6,gene_name_changed_by_hgnc,260,8,0.030769,0.019495,1.596639,0.168409,0.974946
8,Excitatory_ExcL6CT_Cluster_14,gene_name_changed_by_hgnc,41,2,0.048780,0.020377,2.465385,0.211465,0.974946
9,Excitatory_ExcL34RORBCUX2_Cluster_6,gene_name_changed_by_hgnc,104,4,0.038462,0.020000,1.960000,0.170450,0.974946
10,Excitatory_ExcNRGN_Cluster_15,gene_name_changed_by_hgnc,40,2,0.050000,0.020367,2.531579,0.203714,0.974946
14,Inhibitory_InhLAMP5NRG1Rosehip_Cluster_12,gene_name_changed_by_hgnc,46,2,0.043478,0.020429,2.179545,0.250609,0.974946


In [24]:
category = "gene_name_changed_by_hgnc"

results = []

for agm_id in updated_df["AGM_ID"].dropna().unique():

    agm_df = updated_df.loc[updated_df["AGM_ID"] == agm_id].copy()

    changed_to_protein_coding = agm_df.loc[
        (agm_df[category] == True) &
        (agm_df["biotype"] == "protein_coding")
    ]

    results.append({
        "AGM_ID": agm_id,
        "n_genes": agm_df["gene_og"].nunique(),
        "n_changed_to_protein_coding": changed_to_protein_coding["gene_og"].nunique(),
        "changed_to_protein_coding_genes": ", ".join(
            changed_to_protein_coding["gene_og"].dropna().unique()
        )
    })

res_df = pd.DataFrame(results)

res_df["prop_changed_to_protein_coding"] = (
    res_df["n_changed_to_protein_coding"] / res_df["n_genes"]
)

res_df = res_df.sort_values("n_changed_to_protein_coding", ascending=False)

res_df

,AGM_ID,n_genes,n_changed_to_protein_coding,changed_to_protein_coding_genes,prop_changed_to_protein_coding
4,Excitatory_ExcL45RORBGABRG1_Cluster_5,199,4,"CTGF, BTBD11, MYLPF, SLC9A3R1",0.020101
7,Excitatory_ExcL35RORBPLCH1_Cluster_14,221,4,"SEPT10, HIST1H2AC, SLC9A3R1, FAM69C",0.018100
9,Excitatory_ExcL34RORBCUX2_Cluster_6,104,4,"FAM19A1, FAM160A1, MARCH1, FAM19A2",0.038462
13,Inhibitory_InhLAMP5NRG1Rosehip_Cluster_1,163,4,"H2AFZ, C9orf16, ATP5MPL, H3F3B",0.024540
23,Immune_Micall_Cluster_0,222,3,"MPP6, FAM19A2, FAM19A5",0.013514
20,Oligodendrocytes_Oli_Cluster_3,136,3,"FAM19A4, SPATA5, C11orf49",0.022059
15,Inhibitory_InhLAMP5NRG1Rosehip_Cluster_6,260,2,"CCDC173, FAM19A4",0.007692
2,Excitatory_ExcL45RORBGABRG1_Cluster_8,63,2,"MARCH1, FAM19A2",0.031746
8,Excitatory_ExcL6CT_Cluster_14,41,2,"FAM19A1, FAM19A2",0.048780
22,Immune_Micall_Cluster_17,85,1,BTBD11,0.011765
